# Clase 4 · Clase sincrónica
### Python para Economistas

### Tres mensajes importantes:

1) Algna duda del cuestionario de los videos de la clase 4?
2) Tomense 2 mins para completar [este cuestionario](https://forms.gle/9Z26VWNknY7KaWpi9) para saber que tan fácil o difícil les resulto la primer tarea.
3) Recuerden siempre terminar de ver los videos antes de la clase por Zoom y completar [el cuestionario](https://forms.gle/WakMsqrmcncMw5QH9) que cierra los miercoles

### Objetivos de hoy
Al final de la clase deberían poder:

- transformar datos **dentro de cada grupo** con `transform` (por ejemplo, la participación de cada región en el total del período) y verificar el resultado;
- pasar una tabla de formato largo a ancho con `pivot`, y saber cuál de los dos formatos necesita cada herramienta;
- hacer gráficos con `seaborn` desde un `DataFrame` (`lineplot` con `hue`, `heatmap`) y conectarlos con un **widget**;
- aplicar expresiones regulares a una columna entera con `.str.contains`, `.str.extract` y `.str.replace`, usando **grupos con nombre** y `\b`;
- construir un buscador con widgets que maneje patrones inválidos con `try` y `except`;
- **validar** una clasificación hecha con expresiones regulares contra un dato conocido y decidir si cada desacuerdo es un error del patrón, un error de los datos o una diferencia de definición;
- corregir nombres mal escritos con `rapidfuzz`: sobre los **valores distintos**, con una tabla que se pueda revisar y un **umbral** (`score_cutoff`);
- crear un **entorno** en VS Code (`conda` o `venv`), instalar librerías desde un `requirements.txt` y saber qué cambia en Colab;
- usar IA para **entender un patrón**, símbolo por símbolo, sin delegarle la escritura del patrón.

> **Regla de trabajo de hoy con IA**
>
> En las prácticas, rotan quién escribe el código para que no sea el mismo todas las clases.
> Las celdas que dicen **Predecir** se responden **antes** de ejecutar.
>
> 🔴 En los ejercicios de programación primero vamos a **pensar y escribir la lógica por nuestra cuenta, en equipo y sin IA**.  
> Queremos ejercitar la capacidad de traducir un problema a pasos computacionales.
>
> 🟡 Más adelante habrá una actividad señalada explícitamente como **“Ahora sí: usar IA”**, donde la usaremos para interpretar un error y practicar una buena forma de pedir ayuda.


## GitHub: ¿Como actualizar el repositorio para acceder al nuevo material de cada clase?

Esto lo repetiran al inicio de cada clase

##### **I. Moverse a la carpeta del repositorio, que fue creada como resultado del comando anterior:**
```
cd Desktop/HerramientasComputacionales2026/
```

##### **II. Desde la terminal, ubíquense en la carpeta del repositorio y ejecuten:**

```bash
git status
```

Si aparece: ```nothing to commit, working tree clean``` . Sino, ver paso V.

Pueden hacer un pull para traer los últimos cambios de GitHub:

```bash
git pull
```

##### **III. Deshacer los cambios que puedan haber hecho en el repositorio para evitar conflictos con la versión de GitHub:**

Si `git status` muestra archivos modificados y no saben si quieren conservarlos, **no sigan automáticamente**. Primero hagan una copia de sus archivos editados y luego corran el siguiente comando para volver a tener su repositorio de clase local igual que el de la nube.

**Atención:** este comando puede borrar cambios locales.

```bash
git reset --hard
```
Y luego si hagan el

```bash
git pull
```

**Nota para próximas clases**: Recuerden que si editaron los archivos originales que había en el repositorio tienen que crear una copia y cambiarle el nombre para no tener conflicto al hacer el pull.


## 1. Calentamiento: las pausas y el cuestionario

En el video había dos pausas para predecir y tres preguntas de cuestionario. Ejecuten cada celda y comparen con lo que habían escrito.

**1.1 Primera pausa.** ¿Cuántas filas devuelve cada línea?

In [ ]:
import re
import pandas as pd

p = pd.DataFrame({"g": ["a", "a", "a", "b", "b"],
                  "v": [ 1,   2,   3,   10,  20]})
print(p.groupby("g")["v"].agg(['mean']).shape)                                  # (1) agg: una fila por grupo
print(p.groupby("g")["v"].transform("mean").shape)                    # (2) transform: las mismas filas
print(p.groupby("g").filter(lambda d: d["v"].mean() > 5).shape)          # (3) filter: grupos enteros

**1.2 Pregunta 1.** ¿Qué `shape` tiene el resultado del `melt`? ¿Por qué la columna `value` queda con tipo `object`?

In [ ]:
df_p1 = pd.DataFrame({
    "Nombre": ["Mabel", "Juan", "Héctor"], "Apellido": ["García", "Pérez", "Rossi"],
    "Edad": [40, 50, 60],
    "Dirección": ["San Martín 900", "Belgrano 1770", "Sarmiento 1888"],
    "Ciudad": ["La Plata", "Rosario", "Santa Rosa"]})
largo_p1 = df_p1.melt(id_vars=["Nombre", "Apellido"])
print(largo_p1.shape)
print(largo_p1.dtypes)

**1.3 Pregunta 3.** Probamos las respuestas con casos que **deben** coincidir y casos que **no**. En (b) la consigna admite dos lecturas: ¿las `a` van todas juntas, o cada `a` puede tener sus dígitos?

In [ ]:
def probar(patron, deben, no_deben):
    for texto in deben:
        print("debe   ", "✔" if re.search(patron, texto) else "✘", repr(texto))
    for texto in no_deben:
        print("no debe", "✔" if not re.search(patron, texto) else "✘", repr(texto))

print("(a)  a.*b$")
probar(r"a.*b$", deben=["ab", "xxaYYb", "a---b"], no_deben=["ba", "abc", "b"])
print("(b)  a+\\d*")
probar(r"a+\d*", deben=["a", "aaa123", "xa7"], no_deben=["zzz", "123", ""])

**Predecir.** ¿Qué devuelven estas dos líneas? La segunda usa un grupo sin captura con un cuantificador: repite el bloque `a` seguido de dígitos.

In [ ]:
print(re.findall(r"a+\d*", "a125a1324a123"))
print(re.findall(r"(?:a\d*)+", "a125a1324a123"))

**1.4 Segunda pausa y un símbolo nuevo: `\b`.** Para quedarse con los años alcanzaba `\d{4}`. Pero miren qué pasa con un número de ley:

In [ ]:
texto = "La Ley 27742 se sancionó en 2024"
print(re.findall(r"\d{4}", texto))
print(re.findall(r"\b\d{4}\b", texto))

`\b` es un **límite de palabra**: la posición entre un carácter de palabra y otro que no lo es. No consume caracteres. `\b\d{4}\b` exige que los cuatro dígitos sean un número entero, no un pedazo de un número más largo. Lo van a necesitar en el TP.

## 2. Entornos

### ¿Qué es un entorno y para qué sirve?

Un **entorno** es una carpeta con su propio intérprete de Python y sus propias librerías, separada del resto de la computadora. Sirve para tres cosas:

1. **Que un proyecto no rompa a otro.** La tesis puede necesitar pandas 2.2 y el curso pandas 3. Cada proyecto tiene su entorno, con sus versiones.
2. **Reproducibilidad.** Un archivo `requirements.txt` (o `environment.yml`) lista las librerías y sus versiones. Con ese archivo, cualquiera, incluido su director de tesis o ustedes mismos dentro de dos años, puede recrear el entorno y obtener los mismos resultados. Es el equivalente de dejar anotada la versión de Stata y los `ado` que usaron.
3. **Instalar sin miedo.** Si una instalación sale mal, se borra el entorno y se crea de nuevo. La instalación de base de Anaconda queda intacta.

La próxima clase usamos `geopandas`, una librería con muchas dependencias que conviene instalar en un entorno propio. Y desde el TP4, **cada entrega tiene que incluir su `requirements.txt`**.

### Opción A (recomendada): VS Code con `conda`

Ya tienen Anaconda instalado desde la clase 0.

1. Abran en VS Code la carpeta del repositorio (**File → Open Folder**).
2. Abran la paleta de comandos (`Ctrl+Shift+P`, o `Cmd+Shift+P` en Mac) y ejecuten **Python: Create Environment**.
3. Elijan **Conda**, después una versión de Python (por ejemplo, 3.12).
4. VS Code crea el entorno dentro de la carpeta (`.conda`) y lo selecciona para ese proyecto.
5. Abran una terminal **dentro de VS Code** (**Terminal → New Terminal**): ya queda con el entorno activado. Instalen las librerías de la clase:
   ```
   pip install -r Clase4/requirements.txt
   ```
6. En el notebook, arriba a la derecha, **Select Kernel** → **Python Environments** → elijan el entorno nuevo. Si VS Code ofrece instalar `ipykernel`, acepten: es lo que permite usar el entorno desde un notebook.

En versiones recientes, VS Code también tiene un panel de entornos en la barra lateral de Python (**Environment Managers**), con un botón **+** que crea un entorno con valores por defecto. Hace lo mismo que los pasos 2 a 4.

### Opción B: desde la terminal

Con `conda` (en la terminal de Anaconda):
```
conda create -n pythecon python=3.12
conda activate pythecon
pip install -r Clase4/requirements.txt
```
Con `venv`, que viene con Python:
```
python -m venv .venv
source .venv/bin/activate            # Mac y Linux
.venv\Scripts\activate               # Windows
pip install -r Clase4/requirements.txt
```
Para salir del entorno: `conda deactivate` o `deactivate`.

**No suban la carpeta del entorno a GitHub.** Agreguen `.venv/` y `.conda/` a un archivo `.gitignore`. Lo que se sube es el `requirements.txt`, no las librerías instaladas.

### ¿Y en Colab?

Colab **no tiene entornos** como los de su computadora. Cada sesión es una máquina virtual nueva, con cientos de librerías ya instaladas por Google. Lo que instalen con `%pip install` dura hasta que la sesión se desconecta o se reinicia el *runtime*: la próxima vez hay que volver a instalarlo. Por eso, en Colab:

- La primera celda del notebook instala lo que falta: `%pip install rapidfuzz` (o `%pip install -r requirements.txt` si subieron el archivo).
- Las versiones preinstaladas cambian cuando Google actualiza el *runtime*. Para fijarlas, **Runtime → Change runtime type → Runtime version** permite elegir una versión anterior; Google mantiene cada versión disponible durante un año.

Usen `%pip` y no `!pip` dentro de un notebook: `%pip` instala en el mismo Python que está ejecutando el notebook. Vale también en VS Code y en Jupyter.

**Comprobación.** Ejecuten esta celda con el entorno nuevo. Si alguna librería falta, instálenla y vuelvan a ejecutarla.

In [ ]:
pip freeze

In [ ]:
import sys
import importlib

print("Python", sys.version.split()[0], "en", sys.executable)
for libreria in ["pandas", "seaborn", "matplotlib", "rapidfuzz", "ipywidgets"]:
    try:
        modulo = importlib.import_module(libreria)
        print(f"  {libreria:12} {modulo.__version__}")
    except ImportError:
        print(f"  {libreria:12} FALTA: pip install {libreria}")

In [ ]:
pip install rapidfuzz

Con el entorno andando, guarden las versiones exactas para poder reproducirlo:
```
pip freeze > requirements_completo.txt
```
`pip freeze` lista **todo** lo instalado, incluidas las dependencias de las dependencias. Para un proyecto conviene un `requirements.txt` corto, escrito a mano, con las librerías que el código importa y sus versiones (`pandas==3.0.5`).

## 3. Práctica 1 en salas: potencia instalada por región

Volvemos a la base de potencia instalada de la clase 3.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

columnas = ["periodo", "central", "agente", "agente_descripcion", "region",
            "fuente_generacion", "tecnologia", "potencia_instalada_mw"]
df = pd.read_csv("sample_data/potencia_instalada_mod.csv", sep="|", encoding="latin1")[columnas]
df = df.drop_duplicates()                                    # los meses duplicados (TP3)
df["fecha"] = pd.to_datetime(df["periodo"], format="%d/%m/%Y %H:%M")
df = df[df["region"] != "(en blanco)"]                       # 24 filas sin región
print(df.shape, df["fecha"].nunique(), "períodos")

**3.1** Construyan `por_region`: la potencia instalada total para cada combinación de `fecha` y `region`, con las dos como columnas comunes.

**Predecir:** ¿cuántas filas debería tener, como máximo? (47 períodos y 10 regiones.) ¿Por qué podría tener menos?

In [ ]:
# Su código acá

**3.2 `transform`.** Agreguen a `por_region` la columna `participacion`: el porcentaje que representa cada región en la potencia total del país **en ese mismo período**.

_Pista: la potencia total del período tiene que quedar repetida en cada fila de ese período. ¿Cuál de los tres métodos del video hace eso?_

**Verifiquen** que, en cada período, las participaciones sumen 100.

In [ ]:
# Su código acá

**3.3 `pivot`.** Pasen `por_region` a formato ancho: una fila por fecha y una columna por región, con la participación como valor. Guárdenlo en `ancho`.

**Predecir:** ¿qué `shape` va a tener? ¿Va a tener valores faltantes? ¿Dónde?

In [ ]:
# Su código acá

**3.4 Un gráfico de líneas con `seaborn`.** Con `por_region` (formato largo), grafiquen la participación de tres regiones a lo largo del tiempo, una línea por región. Elijan las regiones con `isin`.

Título y nombres de los ejes. Debajo, en una línea: ¿qué región ganó participación?

In [ ]:
# Su código acá

**3.5 Un mapa de calor.** Calculen la participación **promedio por año** de cada región y muéstrenla con `sns.heatmap`, con regiones en las filas, años en las columnas y el valor escrito en cada celda (`annot=True`, `fmt=".1f"`).

_Pista: `por_region["fecha"].dt.year` da el año. Después hay que agrupar y pasar a formato ancho._

In [ ]:
# Su código acá

**3.6 Un widget para elegir la región.** Escriban una función `graficar_region(region)` que grafique la **potencia instalada** (en MW, no la participación) de una región a lo largo del tiempo, y conéctenla con `interact` y un `Dropdown` con las regiones ordenadas alfabéticamente.

In [ ]:
# Su código acá

**3.7 (Para quienes terminan antes) `filter`.** ¿Cuántas centrales aparecen en los 47 períodos? Guárdenlas en `siempre`, usando `groupby` y `filter`. ¿Qué porcentaje de la potencia de febrero de 2020 corresponde a esas centrales?

In [ ]:
# Su código acá

## 4. Demo: expresiones regulares dentro de pandas

### 4.1 Los métodos `.str`

En el video aplicamos patrones a un *string* por vez. Con una columna de pandas no hace falta un `for`: los métodos de `.str` aplican el patrón a todas las filas.

| Método | Devuelve | Equivale a |
|---|---|---|
| `.str.contains(patron)` | una Series de `True`/`False` (sirve para filtrar) | `re.search` |
| `.str.extract(patron)` | un DataFrame con una columna por grupo de captura | `re.search` + `.group()` |
| `.str.replace(patron, repl, regex=True)` | una Series con los reemplazos | `re.sub` |
| `.str.findall(patron)` | una Series de listas | `re.findall` |

In [ ]:
nombres = pd.Series(["CT AÑATUYA II - SULLAIR", "CT CATAMARCA - SECCO", "PARQUE EOLICO ARAUCO SAPEM",
                     "HIDROELECTRICA EL CHOCON SA", "CT BRAGADO - EMGASUD"])

print(nombres.str.contains(r"^CT\b"))

Con **grupos con nombre**, `(?P<nombre>...)`, cada grupo se convierte en una columna con ese nombre. El `?` después de `+` hace que el cuantificador sea **perezoso**: toma lo mínimo posible, para que el lugar no se coma el guion.

In [ ]:
nombres.str.extract(r"^CT\s+(?P<lugar>.+?)\s*-\s*(?P<operador>.+)$")

Las filas que no coinciden quedan con `NaN`. **Siempre cuenten cuántas filas quedaron sin coincidir y miren algunas**: es la forma de saber qué formatos les faltan.

### 4.2 Un buscador con widgets: el *Martín Fierro*

Volvemos a estructurar el poema como en el video, ahora dentro de una función.

In [ ]:
def estructurar_martin_fierro(ruta):
    with open(ruta, encoding="utf8") as archivo:
        texto = archivo.read().split("End of Project Gutenberg")[0]
    partes = re.split(r"[IVX]+ - .*\n", texto)[1:]
    versos = []
    for parte in partes:
        estrofas = re.split(r"\n\d+\n", parte)[1:]
        lista = []
        for estrofa in estrofas:
            lista.append(estrofa.strip().split("\n"))
        versos.append(lista)
    return versos

versos = estructurar_martin_fierro("archivos/martin_fierro.txt")
print(len(versos), "partes")

Un buscador: se escribe un patrón y muestra en qué parte, estrofa y verso aparece. `enumerate` devuelve, en cada vuelta del `for`, el número de orden y el elemento. El `try` evita que un patrón mal escrito (por ejemplo, un paréntesis sin cerrar) detenga todo.

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

def buscar(patron):
    try:
        regex = re.compile(patron)
    except re.error as error:
        print("Patrón inválido:", error)
        return
    encontrados = 0
    for i, parte in enumerate(versos):
        for j, estrofa in enumerate(parte):
            for k, verso in enumerate(estrofa):
                if regex.search(verso):
                    encontrados += 1
                    if encontrados <= 10:
                        print(f"parte {i + 1}, estrofa {j + 1}, verso {k + 1}: {verso}")
    print(f"{encontrados} versos coinciden con {patron!r}")

interact(buscar, patron=widgets.Textarea(value=r"\bgaucho\b", description="Patrón"));

**El 7 % que quedó pendiente.** En el video, el patrón de la primera y la última palabra funcionaba en el 93 % de los versos. Con el buscador, miremos los que fallan:

In [ ]:
def versos_que_fallan(patron):
    fallan = []
    for parte in versos:
        for estrofa in parte:
            for verso in estrofa:
                if len(re.findall(patron, verso)) != 2:
                    fallan.append(verso)
    return fallan

fallan = versos_que_fallan(r"^\w+|\w+[.,;:]?$")
print(len(fallan), "versos")
fallan[:12]

**Predecir:** ¿qué tienen en común? Escriban un patrón mejorado y prueben cuántos versos quedan afuera.

In [ ]:
fallan_nuevo = versos_que_fallan(r"^\W*\w+|\w+\W*$")
print(len(fallan_nuevo), "versos")
fallan_nuevo

## 5. Práctica 2 en salas: nombres de centrales y fuentes mal escritas

Muchas bases económicas identifican a las unidades por un **nombre escrito a mano**: empresas, municipios, escuelas, barrios. Antes de usarlo hay que limpiarlo y **comprobar** que la limpieza funcionó. Practicamos con los nombres de los agentes de la base de potencia.

**5.1** Construyan `agentes`: una tabla con las combinaciones únicas de `agente`, `agente_descripcion` y `fuente_generacion` de `df`. ¿Cuántas filas tiene? ¿Hay agentes que aparecen con más de una fuente?

In [ ]:
# Su código acá

**5.2 Predecir.** ¿Cuántos nombres contienen `EOLIC`? Anoten un número y después compruébenlo con `.str.contains`.

In [ ]:
# Su código acá

**5.3 Clasificar por el nombre.** Muchos nombres dicen qué tipo de central es: `CT` o `C.T.` (central térmica), `TERMOELECTRICA`, `HIDROELECTRICA`, `C.H.`, `PARQUE EOLICO`, `SOLAR`, `FOTOV`… Agreguen a `agentes` una columna `tipo_nombre` con valores `"Térmica"`, `"Hidráulica"`, `"Eólica"`, `"Solar"` o `"Nuclear"`, según el nombre, y `None` si el nombre no lo dice.

Usen un diccionario `{tipo: patron}` y un `for`. ¿Qué porcentaje de los agentes pudieron clasificar?

In [ ]:
# Su código acá

**5.4 Validar.** La base ya tiene la fuente de cada agente. Comparen su clasificación con `fuente_generacion` usando `pd.crosstab`. Miren **todas** las filas en las que no coinciden y, para cada caso, decidan: ¿el error está en su patrón, en los datos, o en que las dos variables miden cosas distintas?

In [ ]:
# Su código acá

**5.5 *Fuzzy matching*.** El archivo `potencia_instalada_con_errores.csv` es la misma base, pero con la columna `fuente_generacion` tipeada con errores. Vamos a corregirla con la librería `rapidfuzz`.

**(a)** Miren los valores distintos de `fuente_generacion` y cuántas veces aparece cada uno.

In [ ]:
# Su código acá

`rapidfuzz` compara *strings* y devuelve un puntaje de similitud entre 0 y 100. `process.extractOne(texto, opciones)` devuelve la opción más parecida, su puntaje y su posición en la lista.

In [ ]:
from rapidfuzz import fuzz, process

print(fuzz.ratio("Termica", "Térmica"))
print(process.extractOne("Temica", ["Térmica", "Renovable", "Hidráulica", "Nuclear"], scorer=fuzz.ratio))

**(b)** Construyan `correccion`: un DataFrame con **una fila por valor distinto** (no por fila de la base), con las columnas `original`, `corregido`, `puntaje` y `filas`. Es una tabla chica que se puede revisar a ojo.

**(c)** Creen un diccionario `{original: corregido}` y úsenlo con `.map` para corregir la columna.

**(d) Verifiquen.** Las frecuencias corregidas tienen que coincidir con las de `potencia_instalada_mod.csv`. Escriban esa comprobación con un `assert`.

In [6]:
# Su código acá

**(e) Predecir.** Supongamos que en la lista de fuentes válidas nos olvidamos de `"Nuclear"`. ¿Qué devuelve `extractOne` para `"Nuclear"`? ¿Qué cambia con `score_cutoff=70`?

In [7]:
# Su código acá

`extractOne` **siempre** devuelve la opción más parecida, aunque sea muy distinta. Sin un umbral, 150 filas nucleares habrían pasado a ser hidráulicas sin ningún aviso. Con `score_cutoff`, devuelve `None` cuando ninguna opción alcanza el puntaje mínimo, y esos casos se revisan a mano.

**Dos reglas para el TP:** corrijan sobre los **valores distintos** (no sobre las filas), así la tabla de corrección se puede auditar; y no confíen en el puntaje solo: miren la tabla.

**5.6 (Para quienes terminan antes) Grupos con nombre.** Muchas centrales térmicas chicas tienen nombres como `CT AÑATUYA II - SULLAIR`: el lugar y la empresa que opera la central, que suele ser una empresa de alquiler de generación. Extraigan `lugar` y `operador` con `.str.extract` y grupos con nombre. ¿Cuántos nombres tienen ese formato? ¿Qué operadores aparecen más?

In [8]:
# Su código acá

## 6. Presentación del TP4

El TP4 trabaja con `domicilios.csv`: 26.502 domicilios de la ciudad de San Luis, cargados a mano por distintas personas a lo largo de los años. Hay que extraer **barrio, manzana y casa** con expresiones regulares, medir cuántos se pudieron estructurar, revisar una muestra a ojo y unificar los nombres de los barrios con *fuzzy matching*. Todo lo que practicamos hoy (`.str.contains`, `.str.extract`, grupos con nombre, `\b`, el buscador con widgets, la tabla de corrección con `rapidfuzz`, validar y mirar los desacuerdos) es lo que van a necesitar.

**Novedad:** el repositorio de la entrega tiene que incluir su `requirements.txt`. Para corregir, vamos a crear un entorno nuevo con ese archivo y ejecutar el notebook.

Antes de escribir código, abran el archivo y **mírenlo**: lean cien domicilios y anoten de cuántas maneras distintas está escrita la manzana.